# Employment Tribunal Judgment Calibration Pipeline

This notebook runs the calibration pipeline with explicit controls for API execution, inputs, and saved outputs.

## 1. Execution Controls

Start here. `RUN_LLM` controls whether the notebook calls the model API. Keep it `False` for setup/dry runs; change it to `True` only when you are ready to spend API tokens.

In [31]:
from pathlib import Path
import getpass
import importlib
import os
import sys

# === API EXECUTION SWITCH ===
RUN_LLM = True          # Change to True to call the model API
SAVE_OUTPUTS = True
USE_CACHE = True
RUN_WS_TAGGING = True  # Uses input/prompts/prompt with the controlled dictionary

# === MODEL/API CONFIGURATION ===
API_PROVIDER = "openai"  # "openai", "anthropic", or "local"
MODEL_NAME = "gpt-5.4"  # Chat Completions model; change only to a valid OpenAI model ID
MAX_COMPLETION_TOKENS = 12000  # WS tagging needs room for complete JSON

# === INPUT BOUNDS ===
MAX_INPUT_FILE_BYTES = 5_000_000
MAX_WS_CHARS = 80_000
MAX_JUDGMENT_CHARS = 120_000
MAX_DICTIONARY_THEMES = 20
MAX_REPAIR_ATTEMPTS = 3

# === DISPLAY BOUNDS ===
MAX_OUTPUT_PREVIEW_CHARS = 2_000
MAX_ERROR_PREVIEW = 10

PROJECT_ROOT = Path.cwd()
SRC_PATH = str(PROJECT_ROOT / "src")
if SRC_PATH in sys.path:
    sys.path.remove(SRC_PATH)
sys.path.insert(0, SRC_PATH)

import config as config_module
import io_utils as io_utils_module
import text_extract as text_extract_module
import dictionary_loader as dictionary_loader_module
import dictionary_runner as dictionary_runner_module
import llm_client as llm_client_module
import calibration_runner as calibration_runner_module
import validators as validators_module
import repair_runner as repair_runner_module
import compression_runner as compression_runner_module

config_module = importlib.reload(config_module)
io_utils_module = importlib.reload(io_utils_module)
text_extract_module = importlib.reload(text_extract_module)
dictionary_loader_module = importlib.reload(dictionary_loader_module)
dictionary_runner_module = importlib.reload(dictionary_runner_module)
llm_client_module = importlib.reload(llm_client_module)
calibration_runner_module = importlib.reload(calibration_runner_module)
validators_module = importlib.reload(validators_module)
repair_runner_module = importlib.reload(repair_runner_module)
compression_runner_module = importlib.reload(compression_runner_module)

Config = config_module.Config
ensure_dirs = io_utils_module.ensure_dirs
read_text_file = io_utils_module.read_text_file
write_json = io_utils_module.write_json
write_text = io_utils_module.write_text
make_run_id = io_utils_module.make_run_id
make_source_slug = io_utils_module.make_source_slug
load_text = text_extract_module.load_text
load_dictionary = dictionary_loader_module.load_dictionary
validate_dictionary = dictionary_loader_module.validate_dictionary
run_ws_tagging = dictionary_runner_module.run_ws_tagging
LLMClient = llm_client_module.LLMClient
run_calibration = calibration_runner_module.run_calibration
CalibrationValidationContext = validators_module.CalibrationValidationContext
validate_calibration_output = validators_module.validate_calibration_output
repair_calibration_output = repair_runner_module.repair_calibration_output
run_compression = compression_runner_module.run_compression

REQUIRED_ENV_BY_PROVIDER = {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "local": None,
}

required_env = REQUIRED_ENV_BY_PROVIDER.get(API_PROVIDER)
if RUN_LLM and required_env and not os.environ.get(required_env):
    os.environ[required_env] = getpass.getpass(f"{required_env}: ")

run_id = make_run_id()
config = Config.default(run_id)
config.api_provider = API_PROVIDER
config.model_name = MODEL_NAME
config.cache_enabled = USE_CACHE
config.max_repair_attempts = MAX_REPAIR_ATTEMPTS
config.max_tokens = MAX_COMPLETION_TOKENS

case_slug = make_source_slug(config.judgment_path)
ensure_dirs(config)

print(f"Run ID: {run_id}")
print(f"RUN_LLM: {RUN_LLM}")
print(f"Provider/model: {config.api_provider} / {config.model_name}")
print(f"Judgment path: {config.judgment_path}")
print(f"Output case slug: {case_slug}")
print(f"Max completion tokens: {config.max_tokens}")
print(f"API key set: {bool(os.environ.get(required_env)) if required_env else 'not required'}")
print(f"Cache enabled: {config.cache_enabled}")
print(f"Run WS tagging prompt: {RUN_WS_TAGGING}")

Run ID: 20260430_205452
RUN_LLM: True
Provider/model: openai / gpt-5.4
Judgment path: /home/hello/Projects/Calibrator/input/judgments/Mr_B_Burke_v_Thomas_Contracting_Ltd_and_Thomas_Plant_Hire_Ltd_-_2414977_2018_-_Reserved.pdf
Output case slug: Mr_B_Burke_v_Thomas_Contracting_Ltd_and_Thomas_Plant_Hire_Ltd_-_2414977_2018_-_Reserved
Max completion tokens: 12000
API key set: True
Cache enabled: True
Run WS tagging prompt: True


## 2. Input Checks

The notebook raises an error if files are missing or above the configured input limits.

In [32]:
def require_file_within_limit(path: Path, max_bytes: int) -> None:
    if not path.exists():
        raise FileNotFoundError(path)
    size = path.stat().st_size
    if size > max_bytes:
        raise ValueError(f"{path} is {size:,} bytes, above limit {max_bytes:,}")

input_paths = [
    config.ws_path,
    config.judgment_path,
    config.dictionary_path,
    config.ws_tagging_prompt_path,
    config.calibration_prompt_path,
    config.repair_prompt_path,
    config.compression_prompt_path,
]

for input_path in input_paths:
    require_file_within_limit(input_path, MAX_INPUT_FILE_BYTES)

print("Input files exist and are within byte limits")

Input files exist and are within byte limits


## 3. Input Loading

PDF extraction uses `output/cache/text` and is keyed by source path, size, and modified time. Repeated notebook runs avoid re-extracting unchanged PDFs.

In [33]:
ws_text = load_text(config.ws_path, config.cache_root / "text", config.cache_enabled)
judgment_text = load_text(config.judgment_path, config.cache_root / "text", config.cache_enabled)

if len(ws_text) > MAX_WS_CHARS:
    raise ValueError(f"WS text is {len(ws_text):,} chars, above limit {MAX_WS_CHARS:,}")
if len(judgment_text) > MAX_JUDGMENT_CHARS:
    raise ValueError(f"Judgment text is {len(judgment_text):,} chars, above limit {MAX_JUDGMENT_CHARS:,}")

print(f"WS text: {len(ws_text):,} chars")
print(f"Judgment text: {len(judgment_text):,} chars")

WS text: 65,472 chars
Judgment text: 21,849 chars


In [34]:
ws_tagging_prompt = read_text_file(config.ws_tagging_prompt_path)
calibration_prompt = read_text_file(config.calibration_prompt_path)
repair_prompt = read_text_file(config.repair_prompt_path)
compression_prompt = read_text_file(config.compression_prompt_path)

# Existing dictionary is used for dry runs and as fallback when generation is disabled.
dictionary = load_dictionary(config.dictionary_path)
validate_dictionary(dictionary)
validation_context = CalibrationValidationContext.from_dictionary(dictionary)

theme_count = len(dictionary.get("ws_theme_dictionary", []))
if theme_count > MAX_DICTIONARY_THEMES:
    raise ValueError(f"Dictionary has {theme_count} themes, above limit {MAX_DICTIONARY_THEMES}")

print(f"Existing dictionary themes: {theme_count}")
print(f"WS tagging prompt loaded: {config.ws_tagging_prompt_path}")
print("Calibration, repair, and compression prompts loaded")

Existing dictionary themes: 20
WS tagging prompt loaded: /home/hello/Projects/Calibrator/input/prompts/prompt
Calibration, repair, and compression prompts loaded


## 4. Run Code

This section performs model calls only when `RUN_LLM = True`. LLM responses are cached in `output/cache/llm` by provider, model, prompt, and payload.

In [35]:
llm_client = LLMClient(
    provider=config.api_provider,
    model=config.model_name,
    temperature=config.temperature,
    max_tokens=config.max_tokens,
    cache_dir=config.cache_root / "llm",
    cache_enabled=config.cache_enabled,
)

print(f"LLM client: {config.api_provider} / {config.model_name}")

LLM client: openai / gpt-5.4


In [36]:
calibration = None
validated_calibration = None
reinforcement_plan = None
ws_tagging = None
errors = []
repair_attempts = 0

if RUN_LLM:
    if RUN_WS_TAGGING:
        ws_tagging = run_ws_tagging(ws_text, dictionary, ws_tagging_prompt, llm_client)
        print(f"WS tagging completed with {len(ws_tagging.get('theme_mappings', []))} theme mappings")
    else:
        print("WS tagging skipped")

    calibration = run_calibration(ws_text, judgment_text, dictionary, calibration_prompt, llm_client)
    errors = validate_calibration_output(calibration, context=validation_context)
    validated_calibration = calibration

    while errors and repair_attempts < config.max_repair_attempts:
        repair_attempts += 1
        validated_calibration = repair_calibration_output(
            validated_calibration,
            errors,
            dictionary,
            repair_prompt,
            llm_client,
        )
        errors = validate_calibration_output(validated_calibration, context=validation_context)

    if errors:
        print(f"Validation failed after {config.max_repair_attempts} repair attempts")
        for error in errors[:MAX_ERROR_PREVIEW]:
            print(error)
        raise ValueError(f"Calibration validation failed after {config.max_repair_attempts} attempts")

    reinforcement_plan = run_compression(validated_calibration, dictionary, compression_prompt, llm_client)
    print("Pipeline execution completed")
else:
    print("RUN_LLM is False; skipped WS tagging, calibration, repair, and compression calls")

WS tagging completed with 20 theme mappings
Pipeline execution completed


## 5. Output

This section saves artifacts only after execution and keeps notebook display output bounded.

In [37]:
output_files = {}

if RUN_LLM and SAVE_OUTPUTS:
    extracted_ws_path = config.output_root / "extracted_text" / f"{run_id}_ws_text.txt"
    extracted_judgment_path = config.output_root / "extracted_text" / f"{run_id}_{case_slug}_text.txt"
    ws_tagging_path = config.output_root / "ws_tagging" / f"{run_id}_ws_tagging.json"
    raw_path = config.output_root / "calibration_raw" / f"{run_id}_{case_slug}_calibration_raw.json"
    validated_path = config.output_root / "calibration_validated" / f"{run_id}_{case_slug}_calibration_validated.json"
    reinforcement_path = config.output_root / "compression" / f"{run_id}_{case_slug}_reinforcement_plan.json"

    write_text(extracted_ws_path, ws_text)
    write_text(extracted_judgment_path, judgment_text)
    if ws_tagging is not None:
        write_json(ws_tagging_path, ws_tagging, validate_reload=config.validate_json_writes)
    write_json(raw_path, calibration, validate_reload=config.validate_json_writes)
    write_json(validated_path, validated_calibration, validate_reload=config.validate_json_writes)
    write_json(reinforcement_path, reinforcement_plan, validate_reload=config.validate_json_writes)

    output_files = {
        "Extracted WS": extracted_ws_path,
        "Extracted Judgment": extracted_judgment_path,
        "Raw Calibration": raw_path,
        "Validated Calibration": validated_path,
        "Reinforcement Plan": reinforcement_path,
    }
    if ws_tagging is not None:
        output_files = {"WS Tagging": ws_tagging_path, **output_files}
    print("Outputs saved")
elif RUN_LLM:
    print("SAVE_OUTPUTS is False; execution results were not written")
else:
    print("No outputs to save because RUN_LLM is False")

Outputs saved


In [38]:
print("=== PIPELINE SUMMARY ===")
print(f"Run ID: {run_id}")
print(f"Input WS chars: {len(ws_text):,} / {MAX_WS_CHARS:,}")
print(f"Input judgment chars: {len(judgment_text):,} / {MAX_JUDGMENT_CHARS:,}")
print(f"Cache root: {config.cache_root}")
print("Dictionary source: existing controlled dictionary file")
print(f"WS tagging output: {'created' if ws_tagging is not None else 'not created'}")

if RUN_LLM and validated_calibration is not None and reinforcement_plan is not None:
    case_name = validated_calibration.get("case_metadata", {}).get("case_name", "Unknown")
    num_signals = len(validated_calibration.get("judgment_signals", []))
    num_clusters = len(reinforcement_plan.get("compressed_reinforcement_plan", []))
    print(f"Case: {case_name}")
    print(f"Judgment signals: {num_signals}")
    print(f"Repair attempts: {repair_attempts}")
    print(f"Compressed clusters: {num_clusters}")
    print(f"Validation errors: {len(errors)}")

    if output_files:
        print("Output files:")
        for label, output_path in output_files.items():
            print(f"  {label}: {output_path}")
else:
    print("Model execution skipped. Set RUN_LLM = True in section 1 to run the expensive cells.")

=== PIPELINE SUMMARY ===
Run ID: 20260430_205452
Input WS chars: 65,472 / 80,000
Input judgment chars: 21,849 / 120,000
Cache root: /home/hello/Projects/Calibrator/output/cache
Dictionary source: existing controlled dictionary file
WS tagging output: created
Case: Burke v Thomas Contracting Limited and Thomas Plant Hire Limited
Judgment signals: 11
Repair attempts: 1
Compressed clusters: 5
Validation errors: 0
Output files:
  WS Tagging: /home/hello/Projects/Calibrator/output/ws_tagging/20260430_205452_ws_tagging.json
  Extracted WS: /home/hello/Projects/Calibrator/output/extracted_text/20260430_205452_ws_text.txt
  Extracted Judgment: /home/hello/Projects/Calibrator/output/extracted_text/20260430_205452_Mr_B_Burke_v_Thomas_Contracting_Ltd_and_Thomas_Plant_Hire_Ltd_-_2414977_2018_-_Reserved_text.txt
  Raw Calibration: /home/hello/Projects/Calibrator/output/calibration_raw/20260430_205452_Mr_B_Burke_v_Thomas_Contracting_Ltd_and_Thomas_Plant_Hire_Ltd_-_2414977_2018_-_Reserved_calibration

In [39]:
if errors:
    print(f"Showing first {min(len(errors), MAX_ERROR_PREVIEW)} validation errors")
    for error in errors[:MAX_ERROR_PREVIEW]:
        print(error)

if RUN_LLM and reinforcement_plan is not None:
    preview = str(reinforcement_plan)[:MAX_OUTPUT_PREVIEW_CHARS]
    print(preview)

{'compressed_reinforcement_plan': {'case_metadata': {'case_name': 'Burke v Thomas Contracting Limited and Thomas Plant Hire Limited', 'case_number': '2414977/2018'}, 'plan_metadata': {'source_type': 'validated_calibration_compression', 'ws_rewrite_performed': False, 'new_facts_created': False, 'non_duplicative_clustering_confirmed': True, 'manual_review_required': True}, 'reinforcement_clusters': [{'cluster_id': 'RC01', 'primary_theme_id': 'T14_FAILURE_TO_VERIFY_EVIDENCE', 'cluster_label': 'Overall failure to verify assumptions and pursue obvious exculpatory checks', 'objective': 'Use the external judgment only as contextual support for the existing WS criticism that disciplinary conclusions should not rest on untested assumptions and that obvious verification steps should be taken before adverse findings are made.', 'absorbed_signal_ids': ['JS01', 'JS06'], 'cross_reference_theme_ids': ['T09_MISSING_LOGS', 'T08_NO_QUEUE_TIME_DISTINCTION', 'T10_OBJECTIVE_COUNTER_EVIDENCE'], 'human_revie